# Phase 2: Baselines

1. **Persistence** — predict last observed speed for all horizons
2. **Linear Regression** — per-road linear model on 15-step history
3. **XGBoost** — global model with engineered features

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
from sklearn.linear_model import LinearRegression
import xgboost as xgb
import time

from src import (
    load_train_speeds, load_train_texts, load_test_data,
    load_adjacency, load_roads,
    build_windows, build_features, compute_norm_stats, compute_mse, train_val_split,
    write_submission,
)

In [ ]:
s1, s2 = load_train_speeds()
t1, t2 = load_train_texts()
adj = load_adjacency()
roads = load_roads()

X1, T1, Y1 = build_windows(s1, t1)
X2, T2, Y2 = build_windows(s2, t2)

X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
T_all = T1 + T2

print(f"Total windows: {len(X_all)}")
print(f"X: {X_all.shape}, Y: {Y_all.shape}")

In [ ]:
X_train, _, Y_train, X_val, _, Y_val = train_val_split(X_all, T_all, Y_all, val_frac=0.2)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")

## 1. Persistence Baseline

In [ ]:
last_speeds = X_val[:, -1, :]  # (N_val, 1260)
y_pred_persist = np.stack([last_speeds] * 3, axis=1)  # (N_val, 3, 1260)
mse_persist = compute_mse(y_pred_persist, Y_val)
print(f"Persistence MSE: {mse_persist:.4f}")

## 2. Linear Regression (per road)

In [ ]:
n_roads = 1260
lr_preds = np.zeros_like(Y_val)

t0 = time.time()
for r in range(n_roads):
    Xr = X_train[:, :, r]  # (N_train, 15)
    Yr = Y_train[:, :, r]  # (N_train, 3)
    model = LinearRegression()
    model.fit(Xr, Yr)
    lr_preds[:, :, r] = model.predict(X_val[:, :, r])

mse_lr = compute_mse(lr_preds, Y_val)
print(f"Linear Regression MSE: {mse_lr:.4f}  ({time.time() - t0:.1f}s)")

## 3. XGBoost (global model with engineered features)

In [ ]:
print("Building train features...")
t0 = time.time()
F_train = build_features(X_train, adj, roads)
print(f"Train features: {F_train.shape} ({time.time() - t0:.1f}s)")

Y_train_flat = Y_train.transpose(0, 2, 1).reshape(-1, 3)  # (N*1260, 3)
print(f"Train targets (flat): {Y_train_flat.shape}")

In [ ]:
models = {}
for hi, hname in enumerate(["h5", "h10", "h15"]):
    model = xgb.XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        verbosity=0,
        n_jobs=-1,
    )
    model.fit(F_train, Y_train_flat[:, hi])
    models[hname] = model
    print(f"  {hname} done")

In [ ]:
print("Building val features...")
F_val = build_features(X_val, adj, roads)

xgb_preds = np.zeros_like(Y_val)
for hi, hname in enumerate(["h5", "h10", "h15"]):
    pred_flat = models[hname].predict(F_val)  # (N_val * 1260,)
    xgb_preds[:, hi, :] = pred_flat.reshape(len(X_val), 1260)

mse_xgb = compute_mse(xgb_preds, Y_val)
print(f"XGBoost MSE: {mse_xgb:.4f}")

## Summary

In [ ]:
print(f"{'Model':<30} {'Val MSE':>10}")
print("-" * 42)
print(f"{'Persistence':<30} {mse_persist:>10.4f}")
print(f"{'Linear Regression (per road)':<30} {mse_lr:>10.4f}")
print(f"{'XGBoost (global)':<30} {mse_xgb:>10.4f}")

best = min(mse_persist, mse_lr, mse_xgb)
for name, mse in [("Persistence", mse_persist), ("Linear", mse_lr), ("XGBoost", mse_xgb)]:
    if mse == best:
        print(f"\nBest: {name} ({mse:.4f})")

In [ ]:
per_horizon_xgb = {}
for hi, hname in enumerate(["h5", "h10", "h15"]):
    mse_h = compute_mse(xgb_preds[:, hi:hi+1, :], Y_val[:, hi:hi+1, :])
    per_horizon_xgb[hname] = mse_h
    print(f"XGBoost {hname} MSE: {mse_h:.4f}")

## Generate Submission

Train on all data, predict test set, write `submissions/YYYYMMDD_HHMMSS_xgb_baseline/submission.csv`.

In [ ]:
F_all = build_features(X_all, adj, roads)
Y_all_flat = Y_all.transpose(0, 2, 1).reshape(-1, 3)

models_full = {}
for hi, hname in enumerate(["h5", "h10", "h15"]):
    model = xgb.XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, verbosity=0, n_jobs=-1,
    )
    model.fit(F_all, Y_all_flat[:, hi])
    models_full[hname] = model
    print(f"  {hname} done")

test_hist, _ = load_test_data()
F_test = build_features(test_hist, adj, roads)

test_preds = np.zeros((540, 3, 1260), dtype=np.float32)
for hi, hname in enumerate(["h5", "h10", "h15"]):
    test_preds[:, hi, :] = models_full[hname].predict(F_test).reshape(540, 1260)

write_submission(test_preds, label="xgb_baseline", models=models_full)